# LightGBM CUDA Benchmark Analysis

This notebook combines the CSV outputs produced by `synthetic/lightgbm_cuda_benchmark.py` and the Slurm launcher `slurm/run_LGBM_cuda_synthetic_benchmark.sbatch`.

It focuses on:
- training time by sample size and device,
- prediction time by sample size and device,
- CPU-to-CUDA speedup across repeated runs,
- convergence proxy via `best_iteration`.

Important limitation: the benchmark CSVs do not contain predictive accuracy metrics such as RMSE, NLL, or CRPS. In this notebook, "performance" therefore means runtime behavior and training characteristics, not forecast quality.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (11, 6)

RESULTS_DIR = Path("../results/synthetic/lightgbm_cuda_benchmark")
COMBINED_CSV_PATH = RESULTS_DIR / "combined_benchmark_runs.csv"

csv_files = sorted(RESULTS_DIR.glob("benchmark_*.csv"))
print(f"Found {len(csv_files)} benchmark files in {RESULTS_DIR}")
for path in csv_files[:5]:
    print(" -", path.name)

In [ ]:
if not csv_files:
    raise FileNotFoundError(f"No benchmark CSVs found in {RESULTS_DIR}")

frames = []
for path in csv_files:
    df = pd.read_csv(path)
    df["source_file"] = path.name
    frames.append(df)

runs = pd.concat(frames, ignore_index=True)
if "stabilization" in runs.columns:
    runs["stabilization"] = runs["stabilization"].fillna("None")
runs["error"] = runs["error"].fillna("")

numeric_cols = [
    "n_samples",
    "n_features",
    "n_informative",
    "train_fraction",
    "num_boost_round",
    "early_stopping_rounds",
    "repeat_idx",
    "seed",
    "start_value_seconds",
    "train_seconds",
    "predict_seconds",
    "best_iteration",
    "speedup_vs_cpu",
    "num_threads",
]
for col in numeric_cols:
    if col in runs.columns:
        runs[col] = pd.to_numeric(runs[col], errors="coerce")

display(runs.head())
print("Rows:", len(runs))
print("Devices:", sorted(runs["device"].dropna().unique().tolist()))
print("Sample sizes:", sorted(runs["n_samples"].dropna().astype(int).unique().tolist()))
print("Statuses:\
", runs["status"].value_counts(dropna=False))

In [ ]:
failed_runs = runs.loc[runs["status"] != "ok"].copy()
ok_runs = runs.loc[runs["status"] == "ok"].copy()

print(f"Successful rows: {len(ok_runs)}")
print(f"Failed/skipped rows: {len(failed_runs)}")
if not failed_runs.empty:
    display(
        failed_runs[
            [
                "source_file",
                "n_samples",
                "seed",
                "device",
                "status",
                "probe_status",
                "error",
            ]
        ].sort_values(["n_samples", "seed", "device"])
    )

ok_runs.to_csv(COMBINED_CSV_PATH, index=False)
print(f"Wrote combined successful runs to {COMBINED_CSV_PATH}")

In [ ]:
summary = (
    ok_runs.groupby(["n_samples", "device"], as_index=False)
    .agg(
        repeats=("seed", "nunique"),
        train_mean=("train_seconds", "mean"),
        train_std=("train_seconds", "std"),
        train_median=("train_seconds", "median"),
        predict_mean=("predict_seconds", "mean"),
        predict_std=("predict_seconds", "std"),
        start_value_mean=("start_value_seconds", "mean"),
        best_iteration_mean=("best_iteration", "mean"),
        best_iteration_std=("best_iteration", "std"),
    )
    .sort_values(["n_samples", "device"])
)

display(summary)

In [ ]:
paired = (
    ok_runs.pivot_table(
        index=["n_samples", "seed"],
        columns="device",
        values=["train_seconds", "predict_seconds", "best_iteration", "start_value_seconds"],
        aggfunc="first",
    )
    .sort_index()
)
paired.columns = [f"{metric}_{device}" for metric, device in paired.columns]
paired = paired.reset_index()

paired["train_speedup_cpu_over_cuda"] = paired["train_seconds_cpu"] / paired["train_seconds_cuda"]
paired["predict_speedup_cpu_over_cuda"] = paired["predict_seconds_cpu"] / paired["predict_seconds_cuda"]
paired["best_iteration_delta_cuda_minus_cpu"] = paired["best_iteration_cuda"] - paired["best_iteration_cpu"]

display(paired)

speedup_summary = (
    paired.groupby("n_samples", as_index=False)
    .agg(
        repeats=("seed", "nunique"),
        train_speedup_mean=("train_speedup_cpu_over_cuda", "mean"),
        train_speedup_std=("train_speedup_cpu_over_cuda", "std"),
        predict_speedup_mean=("predict_speedup_cpu_over_cuda", "mean"),
        predict_speedup_std=("predict_speedup_cpu_over_cuda", "std"),
        best_iteration_delta_mean=("best_iteration_delta_cuda_minus_cpu", "mean"),
    )
    .sort_values("n_samples")
)

display(speedup_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

sns.lineplot(
    data=ok_runs,
    x="n_samples",
    y="train_seconds",
    hue="device",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[0],
)
axes[0].set_title("Training Time by Sample Size")
axes[0].set_xlabel("n_samples")
axes[0].set_ylabel("train_seconds")
axes[0].set_xscale("log")

sns.lineplot(
    data=ok_runs,
    x="n_samples",
    y="predict_seconds",
    hue="device",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[1],
)
axes[1].set_title("Prediction Time by Sample Size")
axes[1].set_xlabel("n_samples")
axes[1].set_ylabel("predict_seconds")
axes[1].set_xscale("log")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

sns.lineplot(
    data=paired,
    x="n_samples",
    y="train_speedup_cpu_over_cuda",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[0],
)
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("CPU / CUDA Training Time Ratio")
axes[0].set_xlabel("n_samples")
axes[0].set_ylabel("speedup (cpu_seconds / cuda_seconds)")
axes[0].set_xscale("log")

sns.lineplot(
    data=paired,
    x="n_samples",
    y="best_iteration_delta_cuda_minus_cpu",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[1],
)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Best Iteration Delta (CUDA - CPU)")
axes[1].set_xlabel("n_samples")
axes[1].set_ylabel("best_iteration difference")
axes[1].set_xscale("log")

plt.tight_layout()
plt.show()

display(
    paired[[
        "n_samples",
        "seed",
        "train_seconds_cpu",
        "train_seconds_cuda",
        "train_speedup_cpu_over_cuda",
        "predict_seconds_cpu",
        "predict_seconds_cuda",
        "predict_speedup_cpu_over_cuda",
        "best_iteration_cpu",
        "best_iteration_cuda",
        "best_iteration_delta_cuda_minus_cpu",
    ]].sort_values(["n_samples", "seed"])
)

## Notes

- If `train_speedup_cpu_over_cuda > 1`, CUDA was faster for training.
- If `train_speedup_cpu_over_cuda < 1`, CPU was faster for training.
- `best_iteration` is only a convergence proxy. It is not a forecast-quality metric.
- To analyze predictive quality in the same notebook, extend `synthetic/lightgbm_cuda_benchmark.py` to log metrics such as RMSE, MAE, quantile loss, NLL, or CRPS on the validation split.